In [13]:
import numpy as np
import fhrr_driver_torch as fd
import torch

In [14]:
A_space = fd.fhrr_space(1000)
B_space = fd.fhrr_space(1000)
E_space = fd.fhrr_space(1000)
AE_map = fd.learning_map(A_space, E_space)
BE_map = fd.learning_map(B_space, E_space)

In [15]:
A_R = A_space.init_random_vec()
A_G = A_space.init_random_vec()
A_B = A_space.init_random_vec()
B_R = B_space.init_random_vec()
B_G = B_space.init_random_vec()
B_B = B_space.init_random_vec()
A_vecs = [A_R, A_G, A_B]
B_vecs = [B_R, B_G, B_B]

In [16]:
def ball_game(p, n):
    p = p[:]
    event_outcomes = []
    for i in range(n):
        outcome = np.random.choice(len(p), p=np.array(p)/np.sum(p))
        event_outcomes.append(outcome)
        p[outcome] -= 1
    return event_outcomes

In [17]:
def query(query_space, answer_space, event_space, QE_map, AE_map, query_vector, answer_vectors):
    answer_matrix = torch.concat(answer_vectors, axis=1)
    event_vector = QE_map.forwards(query_vector, normalize = True)
    answer_vector = AE_map.backwards(event_vector)
    pdf = answer_space.get_pdf(answer_vector, answer_matrix)
    return pdf
def query2(query_space, answer_space, event_space, QE_map, AE_map, query_vector, answer_vectors):
    answer_matrix = torch.concat(answer_vectors, axis=1)
    event_vector = QE_map.forwards(query_vector, normalize = False)
    answer_vector = AE_map.backwards(event_vector)
    pdf = answer_space.get_pdf(answer_vector, answer_matrix)
    return pdf

In [18]:
def recall(answer_space, event_space, AE_map, event_vector, answer_vectors):
    answer_matrix = torch.concat(answer_vectors, axis=1)
    answer_vector = AE_map.backwards(event_vector)
    pdf = answer_space.get_pdf(answer_vector, answer_matrix)
    return pdf

In [19]:
num_trials = 1000
p = [2, 5, 3]
n = 2
avg_outcome_a = [0, 0, 0]
avg_outcome_b = [0, 0, 0]
all_events = E_space.init_zeros_vec()
for trial in range(num_trials):
    outcome = ball_game(p, n)
    avg_outcome_a[outcome[0]] += 1/num_trials
    avg_outcome_b[outcome[1]] += 1/num_trials
    trial_event_vector = E_space.init_random_vec()
    all_events = E_space.bundle(all_events, trial_event_vector)
    AE_map.learn(A_vecs[outcome[0]], trial_event_vector)
    BE_map.learn(B_vecs[outcome[1]], trial_event_vector)
print(avg_outcome_a, avg_outcome_b)

[0.21100000000000016, 0.48100000000000037, 0.3080000000000002] [0.20200000000000015, 0.4920000000000004, 0.3060000000000002]


In [20]:
print(query(A_space, A_space, E_space, AE_map, AE_map, A_space.bundle(*A_vecs), A_vecs))
print(query2(A_space, A_space, E_space, AE_map, AE_map, A_space.bundle(*A_vecs), A_vecs))

tensor([[0.2322, 0.4707, 0.2971]])
tensor([[0.2302, 0.4798, 0.2900]])


In [21]:
#print(query(A_space, A_space, E_space, AE_map, AE_map, A_space.bundle(*A_vecs), A_vecs))

In [22]:
print(recall(A_space, E_space, AE_map, E_space.normalize(all_events), A_vecs))
print(recall(A_space, E_space, AE_map, all_events, A_vecs))

tensor([[0.2244, 0.4845, 0.2911]])
tensor([[0.2158, 0.5087, 0.2755]])


In [23]:
print(query(A_space, A_space, E_space, AE_map, AE_map, A_R, A_vecs))
print(query2(A_space, A_space, E_space, AE_map, AE_map, A_R, A_vecs))

tensor([[0.8816, 0.0226, 0.0958]])
tensor([[0.8846, 0.0195, 0.0959]])


In [24]:
print(query(A_space, B_space, E_space, AE_map, BE_map, A_R, B_vecs))
print(query2(A_space, B_space, E_space, AE_map, BE_map, A_R, B_vecs))
print([1/9, 5/9, 3/9])

tensor([[0.1398, 0.5615, 0.2986]])
tensor([[0.1323, 0.5745, 0.2931]])
[0.1111111111111111, 0.5555555555555556, 0.3333333333333333]
